In [13]:
%%bash
module unload cuda/12.3
module load cuda/12.6
module list

Currently Loaded Modulefiles:
 1) cuda/12.6  


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool

ImportError: /home/akshit.sinha/miniconda3/envs/unlr/lib/python3.10/site-packages/torch/lib/../../nvidia/cusparse/lib/libcusparse.so.12: undefined symbol: __nvJitLinkComplete_12_4, version libnvJitLink.so.12

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, **kwargs):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, out_dim)

    def forward(self, data):
        # graph classification
        x, edge_index = data.x, data.edge_index
        x1 = self.conv1(x, edge_index)
        x1 = F.relu(x1)
        x2 = self.conv2(x1, edge_index)
        x2 = F.relu(x2)
        x3 = self.conv3(x2, edge_index)
        
        x = global_mean_pool(x3, data.batch)
        
        return x

In [ ]:
# import a graph classification dataset
from torch_geometric.datasets import TUDataset

dataset = TUDataset(root='data/TUDataset', name='PROTEINS')

# import DataLoader
from torch_geometric.data import DataLoader

train_ratio = 0.7
val_ratio = 0.1

# random split dataset
dataset = dataset.shuffle()
train_dataset = dataset[:int(len(dataset)*train_ratio)]
val_dataset = dataset[int(len(dataset)*train_ratio):int(len(dataset)*(train_ratio+val_ratio))]
test_dataset = dataset[int(len(dataset)*(train_ratio+val_ratio)):]

In [ ]:
# train
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# define model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GCN(dataset.num_features, 64, dataset.num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

def test(model, loader):
    model.eval()
    correct = 0
    for data in loader:
        data = data.to(device)
        output = model(data)
        pred = output.max(dim=1)[1]
        correct += pred.eq(data.y).sum().item()
    return correct / len(loader.dataset)

def train(model, optimizer, criterion, train_loader, val_loader, test_loader):
    for epoch in range(1, 401):
        model.train()
        for data in train_loader:
        
            data = data.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, data.y)
            loss.backward()
            optimizer.step()
        
        train_acc = test(model, train_loader)
        val_acc = test(model, val_loader)
        test_acc = test(model, test_loader)
        print(f'Epoch: {epoch:03d}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

train(model, optimizer, criterion, train_loader, val_loader, test_loader)        